## Step 0 — Verify the Python environment

Before calling the API, confirm that this notebook is using the project-specific Python environment.

This project uses a local Conda environment stored inside the project directory:

`D:\mycode\nsw-ev-data-pipeline\.conda`

Using a dedicated environment helps keep dependencies isolated from other projects.

In [1]:
import sys

In [2]:
print(sys.executable)

D:\mycode\nsw-ev-data-pipeline\.conda\python.exe


## Step 1 — Define the CKAN dataset

Transport for NSW publishes the EV Charging Locations dataset through a CKAN-based open data portal.

In this step, we define:

- the CKAN base URL;
- the dataset ID;
- the API endpoint that will be used later.

The goal is to discover dataset resources through the CKAN API instead of immediately hard-coding a CSV download URL.

In [4]:
import requests

BASE_URL = "https://opendata.transport.nsw.gov.au/data"
DATASET_ID = "be1c4de4-4517-4bd0-8a09-2965ddfc7179"

## Step 2 — Call the `package_show` API

CKAN provides an Action API endpoint called `package_show`.

This endpoint returns metadata for one dataset, including:

- dataset title;
- description;
- metadata timestamps;
- tags;
- available resources;
- resource IDs;
- resource formats;
- resource URLs.

The request uses the dataset ID as a query parameter.

At this stage, we are requesting dataset metadata only. We are not downloading the CSV file yet.

In [6]:
endpoint = f"{BASE_URL}/api/3/action/package_show"

response = requests.get(
    endpoint,
    params={"id": DATASET_ID}
)

## Step 3 — Inspect the HTTP response

Before parsing the response body, check whether the HTTP request succeeded.

A status code of `200` means that the server successfully returned a response.

`response.url` also shows the final request URL generated by `requests`, including the query parameter.

In [8]:
print("Status:", response.status_code)
print("URL:", response.url)

Status: 200
URL: https://opendata.transport.nsw.gov.au/data/api/3/action/package_show?id=be1c4de4-4517-4bd0-8a09-2965ddfc7179


## Step 4 — Parse the JSON response

The CKAN API returns data in JSON format.

`response.json()` converts the JSON response into Python objects, usually dictionaries and lists.

For a successful CKAN Action API response, the top-level structure commonly contains:

- `help`
- `success`
- `result`

Instead of printing the entire JSON object, inspect its type and top-level keys first.

In [10]:
data = response.json()

print("Type:", type(data))
print("Keys:", data.keys())
print("Success:", data["success"])

Type: <class 'dict'>
Keys: dict_keys(['help', 'success', 'result'])
Success: True


In [11]:
data["result"]

{'author': '',
 'author_email': 'opendataprogram@transport.nsw.gov.au',
 'creator_user_id': None,
 'id': 'be1c4de4-4517-4bd0-8a09-2965ddfc7179',
 'isopen': True,
 'license_id': 'cc-by',
 'license_title': 'Creative Commons Attribution',
 'license_url': 'http://www.opendefinition.org/licenses/cc-by',
 'maintainer': 'TfNSW Open Data Hub and Developer Portal',
 'maintainer_email': 'opendataprogram@transport.nsw.gov.au',
 'metadata_created': '2021-02-10T04:00:41',
 'metadata_modified': '2026-07-14T04:33:23.894924',
 'name': 'ev-charging-locations',
 'notes': " >As Electric Vehicles (EVs) become more prevalent, it's essential that EV charging infrastructure quality and accessibility evolves. Our aim is to enhance the driving experience for EV users throughout New South Wales. \r\n\r\nTo support this initiative, we have released an extensive dataset that includes destination (AC) and fast (DC) EV charging stations currently available in NSW, as well as those scheduled for future development. 

## Step 5 — Inspect selected dataset metadata

The `result` object contains the metadata for the CKAN dataset.

Instead of printing the entire dictionary, inspect a small number of useful fields first.

Important dataset-level fields include:

- `id`: unique dataset identifier
- `name`: CKAN dataset name
- `title`: human-readable title
- `metadata_created`: when the dataset metadata was first created
- `metadata_modified`: when the dataset metadata was last modified
- `num_resources`: number of resources currently attached to the dataset

Dataset-level metadata should not be confused with the modification dates of individual resources.

In [12]:
dataset = data["result"]

print("Title:", dataset["title"])
print("Name:", dataset["name"])
print("Dataset ID:", dataset["id"])
print("Created:", dataset["metadata_created"])
print("Modified:", dataset["metadata_modified"])
print("Number of resources:", dataset["num_resources"])

Title: EV Charging Locations
Name: ev-charging-locations
Dataset ID: be1c4de4-4517-4bd0-8a09-2965ddfc7179
Created: 2021-02-10T04:00:41
Modified: 2026-07-14T04:33:23.894924
Number of resources: 4


## Step 6 — Explore dataset resources

A CKAN dataset can contain multiple resources.

Each resource is represented as a Python dictionary inside the dataset's `resources` list.

Useful resource fields include:

- `id`
- `name`
- `format`
- `description`
- `url`
- `created`
- `last_modified`
- `datastore_active`

Before selecting a resource programmatically, inspect all available resources rather than assuming that the first resource is always the correct one.

In [13]:
resources = dataset["resources"]

print("Type:", type(resources))
print("Number of resources:", len(resources))

Type: <class 'list'>
Number of resources: 4


In [15]:
for i, resource in enumerate(resources):
    print(f"\nResource {i}")
    print("Name:", resource.get("name"))
    print("Format:", resource.get("format"))
    print("Description:", resource.get("description"))
    print("Datastore active:", resource.get("datastore_active"))


Resource 0
Name: EV Charging Locations in NSW
Format: CSV
Description: Effective 20 April 2026
Datastore active: False

Resource 1
Name: EV Charging Locations documentation
Format: PDF
Description: Effective 20 April 2026
Datastore active: False

Resource 2
Name: EV Charging Stations in NSW - Not updated 
Format: CSV
Description: This dataset is no longer current, it is published for historic purposes only.
Datastore active: True

Resource 3
Name: NRMA Co-Funded Charger Usage Report - Not updated
Format: XLS
Description: This dataset is no longer current, it is published for historic purposes only..
Charging data (number of charge sessions, total kWh, average kWh, average time) for NRMA co-funded charging stations. Monthly data: September 2020 - October 2022.



Datastore active: False


## Step 7 — Select the EV CSV resource programmatically

The dataset contains multiple resources, including CSV, PDF, historical CSV, and XLS files.

Instead of selecting a resource by its list position, filter the resources using metadata.

For the current EV dataset, the desired resource should:

- use the `CSV` format;
- represent the current EV Charging Locations dataset;
- not be marked as an old or "Not updated" resource.

This makes the selection logic more explicit and less dependent on the order returned by the API.

In [17]:
csv_resources = [
    resource
    for resource in resources
    if resource.get("format", "").upper() == "CSV"
]

print("CSV resources found:", len(csv_resources))

for resource in csv_resources:
    print(
        resource.get("name"),
        "|",
        resource.get("description")
    )

CSV resources found: 2
EV Charging Locations in NSW | Effective 20 April 2026
EV Charging Stations in NSW - Not updated  | This dataset is no longer current, it is published for historic purposes only.


### Filter out historical resources

There are multiple CSV resources, but one is explicitly labelled as an old resource.

Resource selection should therefore consider both file format and descriptive metadata rather than format alone.

In [18]:
current_csv_candidates = [
    resource
    for resource in csv_resources
    if "not updated" not in resource.get("name", "").lower()
]

print("Current CSV candidate:", len(current_csv_candidates))

for resource in current_csv_candidates:
    print("Name:", resource.get("name"))
    print("ID:", resource.get("id"))
    print("URL:", resource.get("url"))

Current CSV candidate: 1
Name: EV Charging Locations in NSW
ID: 7bbb6461-e52d-4fe7-ace4-a15c30198de0
URL: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv


In [20]:
if len(current_csv_candidates) != 1:
    raise ValueError(
        f"Expected exactly one current EV CSV resource,"
        f"found {len(current_csv_candidates)}."
    )

selected_resource = current_csv_candidates[0]

In [21]:
print("Selected resource:", selected_resource["name"])
print("Resource ID:", selected_resource["id"])
print("Format:", selected_resource["format"])
print("URL:", selected_resource["url"])

Selected resource: EV Charging Locations in NSW
Resource ID: 7bbb6461-e52d-4fe7-ace4-a15c30198de0
Format: CSV
URL: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv


## Step 8 — Inspect the selected resource with `resource_show`

The `package_show` endpoint returns metadata for the entire dataset.

CKAN also provides `resource_show`, which retrieves metadata for one specific resource using its resource ID.

This is useful after resource discovery because it allows the selected file to be inspected independently before downloading it.

In [22]:
resource_endpoint = f"{BASE_URL}/api/3/action/resource_show"

resource_response = requests.get(
    resource_endpoint,
    params={"id": selected_resource["id"]},
    timeout = 30
)

print("Status:", resource_response.status_code)
print("URL:", resource_response.url)

Status: 200
URL: https://opendata.transport.nsw.gov.au/data/api/3/action/resource_show?id=7bbb6461-e52d-4fe7-ace4-a15c30198de0


In [23]:
resource_response.raise_for_status()

resource_data = resource_response.json()

print("Success:", resource_data["success"])
print("Result type:", type(resource_data["result"]))

Success: True
Result type: <class 'dict'>


In [24]:
resource_metadata = resource_data["result"]

print("Name:", resource_metadata.get("name"))
print("Format:", resource_metadata.get("format"))
print("Description:", resource_metadata.get("description"))
print("Created:", resource_metadata.get("created"))
print("Last modified:", resource_metadata.get("last_modified"))
print("Datastore active:", resource_metadata.get("datastore_active"))
print("URL:", resource_metadata.get("url"))

Name: EV Charging Locations in NSW
Format: CSV
Description: Effective 20 April 2026
Created: 2024-09-13T01:49:05.536121
Last modified: 2026-04-20T00:33:22.518559
Datastore active: False
URL: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv


## Step 9 — Download the API-discovered resource

The previous steps used the CKAN API to discover and inspect the desired EV charging resource.

The resource URL can now be obtained directly from the API metadata rather than being hard-coded in the notebook.

The downloaded file will be stored in the **Bronze layer**.

The Bronze layer preserves source data as close as possible to the original form. Cleaning and transformation will be performed later in the Silver layer.

The acquisition flow is now:

`Dataset ID → package_show → resource selection → resource_show → resource URL → download`

In [25]:
from pathlib import Path

current_dir = Path.cwd().resolve()

for candidate in (current_dir, *current_dir.parents):
    if (candidate / "README.md").is_file() and (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Project root could not be found.")

print("Project root:", PROJECT_ROOT)

Project root: D:\mycode\nsw-ev-data-pipeline


### Create the Bronze EV directory

Raw source data is stored in the Bronze layer before any cleaning or transformation.

A separate `ev` directory keeps this source independent from other datasets that will be added later, such as ABS spatial boundaries.

In [26]:
EV_BRONZE_DIR = PROJECT_ROOT /"data" / "bronze" /"ev"

EV_BRONZE_DIR.mkdir(parents = True, exist_ok = True)

print("EV Bronze directory:", EV_BRONZE_DIR)

EV Bronze directory: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev


In [27]:
download_url = resource_metadata["url"].strip()

print("Discovered download URL:")
print(download_url)

Discovered download URL:
https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv


In [28]:
from urllib.parse import urlparse

filename = Path(urlparse(download_url).path).name

print("Filename:", filename)

Filename: ev_20251216.csv


In [29]:
output_path = EV_BRONZE_DIR /filename

print("Output path:", output_path)

Output path: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.csv


### Download the resource

The selected resource is downloaded with an HTTP GET request.

Before saving the response, the HTTP status is checked with `raise_for_status()`.

Streaming the response in chunks avoids requiring the entire file to be held in memory at once and is more suitable for larger datasets.

In [33]:
with requests.get(download_url, stream=True, timeout=60) as download_response:
    download_response.raise_for_status()

    
    with output_path.open("wb") as file:
        for chunk in download_response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)

print("Downloaded to:", output_path)

Downloaded to: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.csv


## Step 10 — Perform basic acquisition checks

A successful HTTP request does not automatically guarantee that the expected dataset was downloaded.

Perform a small number of basic checks before continuing:

- the file exists;
- the file is not empty;
- inspect the file size;
- inspect the server-reported content type.

Deeper schema and data-quality validation will be performed later.

In [34]:
print("Exists:", output_path.exists())
print("Size:", output_path.stat().st_size, "bytes")
print("Content-Type:", download_response.headers.get("Content-Type"))

Exists: True
Size: 283033 bytes
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet


In [35]:
with output_path.open("r", encoding="utf-8-sig") as file:
    print(file.readline().strip())
    print(file.readline().strip())

OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
,,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.26224229,150.8901391,Muswellbrook Shire Council,2333,Existing Destination Chargers


## Step 11 — Calculate a SHA-256 checksum

A checksum provides a reproducible fingerprint of the downloaded Bronze file.

If the same source is downloaded again later, the checksum can be compared to determine whether the file bytes have changed.

SHA-256 does not describe the meaning or correctness of the data. It only identifies the exact file content.

In [36]:
import hashlib

sha256 = hashlib.sha256()

with output_path.open("rb") as file:
    for chunk in iter(lambda: file.read(8192), b""):
        sha256.update(chunk)

file_hash = sha256.hexdigest()

print("SHA-256:", file_hash)

SHA-256: 43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a


## Step 12 — Record data provenance

A Bronze dataset should retain information about where it came from.

Useful provenance fields include:

- dataset ID;
- resource ID;
- resource name;
- source URL;
- source metadata timestamps;
- retrieval timestamp;
- local filename;
- file size;
- SHA-256 checksum;
- observed HTTP Content-Type.

This metadata helps explain and reproduce the acquisition process later.

In [43]:
from datetime import datetime, timezone

retrieved_at = datetime.now(timezone.utc).isoformat()

print(retrieved_at)

2026-09-14T10:36:06.056045+00:00


In [45]:
provenance = {
    "dataset_id": dataset["id"],
    "dataset_title": dataset["title"],
    "resource_id": resource_metadata["id"],
    "resource_name": resource_metadata["name"],
    "resource_format": resource_metadata["format"],
    "resource_description": resource_metadata["description"],
    "resource_last_modified": resource_metadata["last_modified"],
    "source_url": download_url,
    "retrieved_at_utc": retrieved_at,
    "local_filename": output_path.name,
    "file_size_bytes": output_path.stat().st_size,
    "sha256": file_hash,
    "http_content_type": download_response.headers.get("Content-Type"),
}

provenance

{'dataset_id': 'be1c4de4-4517-4bd0-8a09-2965ddfc7179',
 'dataset_title': 'EV Charging Locations',
 'resource_id': '7bbb6461-e52d-4fe7-ace4-a15c30198de0',
 'resource_name': 'EV Charging Locations in NSW',
 'resource_format': 'CSV',
 'resource_description': 'Effective 20 April 2026',
 'resource_last_modified': '2026-04-20T00:33:22.518559',
 'source_url': 'https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv',
 'retrieved_at_utc': '2026-09-14T10:36:06.056045+00:00',
 'local_filename': 'ev_20251216.csv',
 'file_size_bytes': 283033,
 'sha256': '43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a',
 'http_content_type': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}

## Step 13 — Save provenance metadata

The provenance dictionary is saved alongside the Bronze dataset as JSON.

Keeping source metadata next to the raw data makes the acquisition process easier to audit and reproduce later.

The JSON file records the exact source, retrieval time, resource metadata, local filename, file size, checksum, and observed HTTP content type.

In [46]:
import json

metadata_path = output_path.with_suffix(".metadata.json")

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(
        provenance,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Metadata saved to:", metadata_path)

Metadata saved to: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.metadata.json


In [47]:
with metadata_path.open("r", encoding="utf-8") as file:
    saved_metadata = json.load(file)

saved_metadata

{'dataset_id': 'be1c4de4-4517-4bd0-8a09-2965ddfc7179',
 'dataset_title': 'EV Charging Locations',
 'resource_id': '7bbb6461-e52d-4fe7-ace4-a15c30198de0',
 'resource_name': 'EV Charging Locations in NSW',
 'resource_format': 'CSV',
 'resource_description': 'Effective 20 April 2026',
 'resource_last_modified': '2026-04-20T00:33:22.518559',
 'source_url': 'https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv',
 'retrieved_at_utc': '2026-09-14T10:36:06.056045+00:00',
 'local_filename': 'ev_20251216.csv',
 'file_size_bytes': 283033,
 'sha256': '43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a',
 'http_content_type': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}

## Step 14 — Test the reusable acquisition module

The exploratory CKAN API logic has been moved into `src/acquisition/ev.py`.

This section tests each reusable function independently before combining them into a complete acquisition pipeline.

In [124]:
import sys

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Source path:", src_path)

Source path: D:\mycode\nsw-ev-data-pipeline\src


In [125]:
from importlib import reload
import acquisition.ev as ev

reload(ev)

<module 'acquisition.ev' from 'D:\\mycode\\nsw-ev-data-pipeline\\src\\acquisition\\ev.py'>

In [126]:
from acquisition.ev import (
    BASE_URL,
    DATASET_ID,
    get_dataset_metadata,
    select_current_csv_resource,
    get_resource_metadata,
    download_resource
)

### Test 1 — Retrieve dataset metadata

Test `get_dataset_metadata()` to confirm that the CKAN `package_show` API returns the expected dataset metadata.

In [127]:
dataset = get_dataset_metadata()

print("Dataset title:", dataset["title"])
print("Number of resources:", dataset["num_resources"])

Dataset title: EV Charging Locations
Number of resources: 4


### Test 2 — Select the current CSV resource

Test `select_current_csv_resource()` to confirm that the module selects the active EV CSV resource rather than documentation or historical resources.

In [128]:
selected_resource = select_current_csv_resource(
    dataset["resources"]
)

print("Name:", selected_resource["name"])
print("Format:", selected_resource["format"])
print("ID:", selected_resource["id"])
print("URL:", selected_resource["url"].strip())

Name: EV Charging Locations in NSW
Format: CSV
ID: 7bbb6461-e52d-4fe7-ace4-a15c30198de0
URL: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv


### Test 3 — Retrieve selected resource metadata

Test `get_resource_metadata()` using the selected resource ID.

The function calls CKAN's `resource_show` endpoint and should return metadata for the selected EV CSV resource, including its format, description, modification timestamp, and download URL.

In [129]:
resource_metadata = get_resource_metadata(
    selected_resource["id"]
)

print("Name:", resource_metadata["name"])
print("Format:", resource_metadata["format"])
print("Description:", resource_metadata["description"])
print("Last modified:", resource_metadata["last_modified"])
print("URL:", resource_metadata["url"].strip())

Name: EV Charging Locations in NSW
Format: CSV
Description: Effective 20 April 2026
Last modified: 2026-04-20T00:33:22.518559
URL: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv


In [130]:
downloaded_path, content_type = ev.download_resource(
    resource_metadata,
    EV_BRONZE_DIR,
)

print("Downloaded path:", downloaded_path)
print("Content-Type:", content_type)

Downloaded path: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.csv
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet


### Test 5 — Calculate the file checksum

Test `calculate_sha256()` on the downloaded Bronze file.

SHA-256 provides a fingerprint of the exact downloaded bytes. If a future download produces a different checksum, the file content has changed.

The checksum verifies file identity, not whether the dataset itself is correct.

In [131]:
file_hash = ev.calculate_sha256(downloaded_path)

print("SHA-256:", file_hash)

SHA-256: 43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a


### Test 6 — Build and save provenance metadata

Test the provenance functions using the downloaded Bronze EV file.

`build_provenance()` combines dataset metadata, resource metadata, file information, retrieval time, and checksum into a Python dictionary.

`save_provenance()` stores this metadata alongside the Bronze CSV as a JSON file.

In [132]:
provenance = ev.build_provenance(
    dataset,
    resource_metadata,
    downloaded_path,
    file_hash,
    http_content_type=content_type,
)

provenance

{'dataset_id': 'be1c4de4-4517-4bd0-8a09-2965ddfc7179',
 'dataset_title': 'EV Charging Locations',
 'resource_id': '7bbb6461-e52d-4fe7-ace4-a15c30198de0',
 'resource_name': 'EV Charging Locations in NSW',
 'resource_format': 'CSV',
 'resource_description': 'Effective 20 April 2026',
 'resource_last_modified': '2026-04-20T00:33:22.518559',
 'source_url': 'https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv',
 'retrieved_at_utc': '2026-09-14T12:33:17.887521+00:00',
 'local_filename': 'ev_20251216.csv',
 'file_size_bytes': 283033,
 'sha256': '43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a',
 'http_content_type': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}

In [133]:
metadata_path = ev.save_provenance(
    provenance,
    downloaded_path,
)

print("Metadata path:", metadata_path)
print("Exists:", metadata_path.exists())

Metadata path: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.metadata.json
Exists: True


In [134]:
import json

with metadata_path.open("r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

saved_metadata

{'dataset_id': 'be1c4de4-4517-4bd0-8a09-2965ddfc7179',
 'dataset_title': 'EV Charging Locations',
 'resource_id': '7bbb6461-e52d-4fe7-ace4-a15c30198de0',
 'resource_name': 'EV Charging Locations in NSW',
 'resource_format': 'CSV',
 'resource_description': 'Effective 20 April 2026',
 'resource_last_modified': '2026-04-20T00:33:22.518559',
 'source_url': 'https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv',
 'retrieved_at_utc': '2026-09-14T12:33:17.887521+00:00',
 'local_filename': 'ev_20251216.csv',
 'file_size_bytes': 283033,
 'sha256': '43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a597109865a',
 'http_content_type': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}

### Test 7 — Run the complete EV acquisition pipeline

Test `acquire_ev_data()` as the end-to-end orchestration function for EV data acquisition.

The function should:

- retrieve dataset metadata;
- select the current CSV resource;
- retrieve resource metadata;
- download the Bronze CSV;
- calculate its SHA-256 checksum;
- build provenance metadata;
- save the provenance JSON file.

A successful run should return both output paths.

In [135]:
ev_file, metadata_file = ev.acquire_ev_data(
    EV_BRONZE_DIR
)

print("EV file:", ev_file)
print("Metadata file:", metadata_file)
print("EV exists:", ev_file.exists())
print("Metadata exists:", metadata_file.exists())

EV file: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.csv
Metadata file: D:\mycode\nsw-ev-data-pipeline\data\bronze\ev\ev_20251216.metadata.json
EV exists: True
Metadata exists: True
